In [24]:
from pathlib import Path

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

In [25]:
!pip install -U langchain langchain-community langchain-text-splitters pypdf pymupdf

Defaulting to user installation because normal site-packages is not writeable


In [26]:
import os
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

### document datastructure
from langchain 

In [27]:
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 3 PDF files to process

Processing: 2508.18473v2.pdf
  ✓ Loaded 16 pages

Processing: chai of verifivcations.pdf
  ✓ Loaded 19 pages

Processing: geometry of truth.pdf
  ✓ Loaded 22 pages

Total documents loaded: 57


In [28]:
all_pdf_documents

[Document(metadata={'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:)', 'creationdate': '', 'author': 'Jiawei Li; Akshayaa Magesh; Venugopal V. Veeravalli', 'doi': 'https://doi.org/10.48550/arXiv.2508.18473', 'license': 'http://arxiv.org/licenses/nonexclusive-distrib/1.0/', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'title': 'Principled Detection of Hallucinations in Large Language Models via Multiple Testing', 'trapped': '/False', 'arxivid': 'https://arxiv.org/abs/2508.18473v2', 'source': '..\\data\\2508.18473v2.pdf', 'total_pages': 16, 'page': 0, 'page_label': '1', 'source_file': '2508.18473v2.pdf', 'file_type': 'pdf'}, page_content='Principled Detection of Hallucinations in Large\nLanguage Models via Multiple Testing\nJiawei Li∗\nDepartment of Electrical and Computer Engineering\nUniversity of Illinois Urbana-Champaign\njiaweil9@illinois.edu\nAkshayaa Magesh*\nMeta\nakshayaamagesh@gmail.com\nV enugopal

In [29]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(all_pdf_documents)
print(f"Total chunks created: {len(chunks)}")

Total chunks created: 268


In [30]:
print(chunks[0].page_content[:500])
print(chunks[0].metadata)

Principled Detection of Hallucinations in Large
Language Models via Multiple Testing
Jiawei Li∗
Department of Electrical and Computer Engineering
University of Illinois Urbana-Champaign
jiaweil9@illinois.edu
Akshayaa Magesh*
Meta
akshayaamagesh@gmail.com
V enugopal V . V eeravalli
Department of Electrical and Computer Engineering
University of Illinois Urbana-Champaign
vvv@illinois.edu
Abstract
While Large Language Models (LLMs) have emerged as powerful foundational
models to solve a variety of 
{'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:)', 'creationdate': '', 'author': 'Jiawei Li; Akshayaa Magesh; Venugopal V. Veeravalli', 'doi': 'https://doi.org/10.48550/arXiv.2508.18473', 'license': 'http://arxiv.org/licenses/nonexclusive-distrib/1.0/', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'title': 'Principled Detection of Hallucinations in Large Language Models via Multiple Testing', 'trapped': '/False', 

In [31]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

In [32]:
embedding_model = HuggingFaceEmbeddings(
 model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 219.22it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [33]:
vectorstore = FAISS.from_documents(chunks, embedding_model)

In [34]:
def retrieve_context(query, vectorstore, k=4):
    docs = vectorstore.similarity_search(query, k=k)
    context = "\n\n".join([doc.page_content for doc in docs])
    return context, docs

In [35]:
def generate_answer(query, context, llm):
    prompt = f"""
Answer the question using ONLY the context below.
If the answer is not present in the context, say "Not found in provided documents".

Context:
{context}

Question:
{query}

Answer:
"""
    return llm(prompt).strip()

In [36]:
def generate_answer(query, context, llm):
    prompt = f"""
Answer the question using ONLY the context below.
If the answer is not present, say "Not found in provided documents".

Context:
{context}

Question:
{query}

Answer:
"""
    response = llm.invoke(prompt)   # response is STRING
    return response.strip()

In [37]:
def hallucination_check_llm(query, answer, context, llm):
    prompt = f"""
You are a factual verifier.

Context:
{context}

Question:
{query}

Answer:
{answer}

Task:
Check whether the answer is factually supported by the context.

Rules:
- If supported → NOT HALLUCINATION
- If unsupported / fabricated → HALLUCINATION

Respond with ONLY one word:
HALLUCINATION or NOT HALLUCINATION
"""
    verdict = llm.invoke(prompt)   # STRING
    return verdict.strip()

In [38]:
pip install openai langchain-openai

Note: you may need to restart the kernel to use updated packages.


In [39]:
from langchain_community.llms import Ollama

llm = Ollama(
    model="llama3",
    temperature=0
)

In [40]:
def rag_hallucination_pipeline(query, vectorstore, llm):
    context, _ = retrieve_context(query, vectorstore)
    answer = generate_answer(query, context, llm)

    # Case 1: model abstains
    if answer.lower().startswith("not found"):
        return {
            "question": query,
            "answer": answer,
            "hallucination_verdict": "NOT ANSWERED (INSUFFICIENT CONTEXT)"
        }

    # Case 2: trivial yes/no answer
    if len(answer.split()) <= 2:
        return {
            "question": query,
            "answer": answer,
            "hallucination_verdict": "UNINFORMATIVE ANSWER (NO GROUNDED CONTENT)"
        }

    # Case 3: normal hallucination check
    verdict = hallucination_check_llm(query, answer, context, llm)

    return {
        "question": query,
        "answer": answer,
        "hallucination_verdict": verdict
    }

In [41]:
query = "What recent techniques have been developed to infer whether a large language model is telling the truth?"

result = rag_hallucination_pipeline(query, vectorstore, llm)

print("Question:", result["question"])
print("Answer:", result["answer"])
print("Hallucination:", result["hallucination_verdict"])

Question: What recent techniques have been developed to infer whether a large language model is telling the truth?
Answer: Recent work has developed techniques for inferring whether a LLM is telling the truth by training probes on the LLM's internal activations.
Hallucination: NOT HALLUCINATION


In [42]:
def is_trivial_answer(answer):
    return len(answer.split()) <= 2  # "Yes", "No", "Maybe"